<img src="https://github.com/moroneyt/MXB301/raw/main/resources/qutlogo.jpg">

# MXB301 Mathematics of AI
# Lesson 2: Gradients

#### Tim Moroney, 2026

A lesson where we learn about loss functions and how to take gradients of our character level language model.

# Package management

We start by installing the required packages.  If you're running on Colab this will just download pre-compiled code.  Otherwise be prepared to wait a few minutes for installation the first time you run. Feel free to read ahead while you wait.

In [ ]:
import Pkg
if haskey(ENV, "COLAB_GPU") # check if we're on Colab
  if !isfile("/content/MXB301_2026_01_CPU.tgz") # check if we've already downloaded
    # download precompiled Julia environment for Colab
    run(`gdown https://drive.google.com/uc\?id=1mT9XFadzdfK8CWb5a7BYLUkd2RTi2eZc`)

    # replace Colab's Julia environment with downloaded version
    run(`rm -rf /root/.julia`)
    run(`tar -xzf MXB301_2026_01_CPU.tgz -C /root`)
  end
else
  # For any other machine we install the packages in the usual way
  Pkg.activate(".")
  Pkg.add(["CairoMakie", "CodecZlib", "ColorSchemes", "ComponentArrays", "CondaPkg",
           "DifferentiationInterface", "Distributions", "Downloads", "FiniteDiff", "ForwardDiff",
           "HTTP", "JLD2", "LaTeXStrings", "LinearAlgebra", "Lux", "MKL", "MLUtils", "NNlib",
           "NLSolversBase", "OneHotArrays", "Optim", "PythonCall", "QuadGK", "Random",
           "SpecialFunctions", "Statistics", "StatsBase", "ToeplitzMatrices", "Zygote"])
end

using CairoMakie
using DifferentiationInterface
using LaTeXStrings
using LinearAlgebra
using Lux
using Random
using SpecialFunctions
using Statistics

using ComponentArrays: ComponentVector
using Distributions: Normal, Exponential
using Downloads: download
using ForwardDiff: Dual, partials
using JLD2: jldopen
using MLUtils: DataLoader, rand_like, randn_like
using NLSolversBase: only_fg
using NNlib: softmax, sigmoid, scatter as scattergrad, conv, ∇conv_filter, ∇conv_data, DenseConvDims
using OneHotArrays: onehot, onehotbatch, onecold
using StatsBase: crossentropy, sample, Weights
using ToeplitzMatrices: Toeplitz, Hankel
using QuadGK: quadgk

import CodecZlib
import ColorSchemes
import FiniteDiff
import HTTP
import MKL
import Optim
import Zygote

# Set the random seed for reproducibility
rng = Random.seed!(0)

# SVG format scales properly in web pages and PDFs
CairoMakie.activate!(type = "svg")

# Review

In the last lesson we developed and analysed a character level language model (CLLM) for predicting the probabilities of the next character from an input sequence of text.  Let's quickly review what we came up with.

## Neural net architecture

Our model comprises an embedding layer, followed by two dense layers with a nonlinear activation in-between, and a softmax output layer. If we wanted to, we could write it all out as a one-line formula, which looks a bit peculiar, but just for fun, that formula is

$$M_p(u) = \textrm{softmax}(W_2\textrm{tanh}.(W_1\textrm{vec}(W_e[:,u])+b_1 )+b_2).$$

That's genuinely all there is to our model.  Here it is in code.

In [ ]:
# Predict probabilities of the next character
function forward(u, p)
    X = p.We[:, u]               # 1. embedding
    v = vec(X)                   # 2. flatten
    z1 = p.W1 * v  + p.b1        # 3. first dense layer
    h1 = tanh.(z1)               # 4. activation
    z2 = p.W2 * h1 + p.b2        # 5. second dense layer
    ŷ = softmax(z2)              # 6. softmax for probabilities
    return ŷ
end

## Towards training

In this lesson we will take a step towards being able to actually train this model.  By "training" we mean updating the values of the model parameters -- the weights and biases stored in the array $p$ -- to improve the model's performance on a set of training data.  We aim to answer the questions:

* How do we quantify the model's performance on the training data?
* How do we adjust the parameters in order to improve this performance?

We will see that the first question is answered by means of the model's _loss function_, while the second question is all about _gradients_.

## Loss function

Currently we have a model that takes an input vector $u$ representing a sequence of characters (stored as integer indices  `'a'` $\Leftrightarrow 1,$ `'b'` $\Leftrightarrow 2,$, and so on), and a parameter vector $p$ storing the weights and biases $W_e$, $W_1$, $b_1$, $W_2$ and $b_2$.  The model outputs a vector of probabilities $\hat{y}$ for the predicted next character in the sequence.

The probabilities that the model outputs are _entirely_ determined by the input $u$ and the parameters $p$.  Let's load our pre-trained parameter vector from last time.

In [ ]:
# Load some pre-trained parameters for this model
url = "https://github.com/moroneyt/MXB301/raw/main/resources/CLLM_pretrained.jld2"
paramfile = jldopen(download(url))
p = paramfile["p"]
close(paramfile)
p # let's see what we have here

## Character set and helper functions for mappings

We'll also set up our character set and mappings between indices and characters.

We have the same example input `"alice said"` as in the last lesson.

In [ ]:
chars = ['a':'z'; ' ']      # we only deal with lowercase text and space
vocab_size = length(chars)  # number of characters in our "vocabulary"

# Mappings between characters and indices (a = 1, b = 2, etc.)
idx_to_char(i) = chars[i]
char_to_idx(c) = findfirst(isequal(c), chars)
string_to_idxs(s) = [char_to_idx(c) for c in s]

u = string_to_idxs("alice said") # sample input

## Model output

The model's output for this input text are the predicted probabilities for the next character in the sequence.  As we saw in the last lesson, using the trained parameter vector $p$, the model made a decent prediction that the most likely next character is space (index 27 has probability of around $0.82$).

In [ ]:
ŷ = forward(u, p)

# Loss functions

One of the most important concepts in AI is the idea of a **loss function**.  Let's motivate the idea by trying a different set of parameters in our model.

## Random parameters

If we were to use a completely random parameter vector, the model's output is very likely to be complete rubbish.  For example:

In [ ]:
prand = randn!(similar(p))  # random parameter values
ŷ2 = forward(u, prand)

## Model prediction

For what it's worth, the model's prediction now for the most likely next character is

In [ ]:
idx_to_char(argmax(ŷ2))

## Ground truth and one-hot vectors

Intuitively we know that this prediction is bad.  You can't continue the text `"alice said"` with the letter `g`.  But how do we quantify this?  We need a way to measure exactly how good or bad the model's prediction is.  For that, we need to compare the output to the _ground truth_.

Suppose that we really had sampled the sequence `"alice said"` from the training text (which was "Alice's Adventures in Wonderland" remember), and that indeed the next character in the text was a space.

We can represent this ground truth in the form of a **one-hot vector** $y$, which is just a fancy way of saying a vector with a 1 in exactly one position and zeros elsewhere: in this case a 1 at index 27, representing the space character.

In [ ]:
y = onehot(char_to_idx(' '), 1:vocab_size)  # one-hot vector representing the space character

## Quantifying the mismatch

Since space is the correct next character in the sequence, the optimal model prediction would be simply $\hat{y} = y$; that is, $\hat{y}$ would also have a 1 in position 27 and zeros elsewhere.  That would correspond to the model being completely confident that the next character is a space.  The extent to which the predicted output $\hat{y}$ disagrees from the ground truth $y$ is the quantity we want to measure in our **loss function**.

Now it's a question of how best to quantify how close our output vector $\hat{y}$ is to the ground truth $y$.  You might guess that using the mean square error
$
\| y - \hat{y} \|^2,
$
would be the right choice.  But actually there is a better choice.  For models that output a probability distribution, such as ours, the preferred measure of error is the **cross entropy**, defined by

$$
\textrm{crossentropy}(y, \hat{y}) = -\sum_{i} y_i\, \log (\hat{y}_i)\,.
$$

This formula comes from information theory, and its somewhat scary-sounding name (anything involving the word "entropy" sounds intimidating right?) reflects these origins.  But in our context, it has a very simple interpretation that isn't scary at all.  

The ground truth $y$ is a one-hot vector, so in reality this "sum" is really only one term:

$$
\textrm{crossentropy}(y, \hat{y}) = -\log (\hat{y}_k)\qquad (y\textrm{ is one-hot at index }k)
$$

which is precisely the negative log-likelihood of the data under the model.  So if we define the loss using cross-entropy, by minimising the loss we are in fact optimising the model's predicted probability for the correct next character.

Notice that if the model is certain and correct about the next character, $\hat{y}_k = 1,$ then the loss is zero.  Conversely if the model has very low confidence in this being the correct next character, $\hat{y}_k \ll 1$, it will be heavily penalised by the negative log, and in fact the loss becomes infinite as $\hat{y}_k \to 0$.



## Trained versus random parameters

Let's give the cross-entropy loss a try with our two model outputs.  First, the output `y` that came from the model using trained parameters, and second with the output `y2` that came from the model using random parameters.

Certainly the loss is much worse with random parameters.

In [ ]:
crossentropy(y, ŷ), crossentropy(y, ŷ2)

## Loss as a function of the parameters

For a fixed training example $(u,y)$, the loss, $\textrm{crossentropy}(y, \hat{y})$, only depends on the parameter values $p$, by way of the model's output $\hat{y} = M_p(u)$.  Hence we can define a scalar loss function

$$
L(p) := \textrm{crossentropy}(y, M_p(u))\,.
$$

From this perspective, the loss value represents how good our current set of parameters $p$ are doing.  If the loss is big, we should alter the parameters $p$ (somehow!) to improve it.

Here we define this function $L$ and confirm that it returns the correct values for the loss, for our two sets of parameters: the pre-trained values, and the random values.

In [ ]:
# Define the loss function
L(p) = crossentropy(y, forward(u, p))

# Confirm we recover the same values as before
L(p), L(prand)

# Training

The whole idea of machine _learning_ is to try to vary the values of our parameters -- the weights and biases -- to reduce the loss by learning from data.  Naturally, in practice our model shouldn't just be trying to reduce the loss on only one training example.  Ideally we should provide the model with many thousands or millions of training examples, and try to reduce the total loss (or average loss, doesn't matter) across all of the training examples.  But we'll keep it simple for now, and ask how we might adjust the parameters in order to do better than the current value of the loss for just this one training example.

## A quick test: nudge the parameters a bit

So pretend we don't have pre-trained parameters; we only have the randomly-initialised parameters.  How can we improve the loss $L(p)$? One thing we could try is just to vary some of the weights, perhaps nudge them one way or another, and observe the effect this has on the loss. Let's do that now.

Maybe the model would perform better if all the embedding matrix weights were a little bigger in magnitude?

And when we try that, as luck would have it, the new loss is slightly lower than before.  But it was just that: luck.  Fiddling around with parameters like this is no way to make real progress.

In [ ]:
prand.We *= 1.1  # increase all embedding matrix weights in magnitude a little
L(prand)         # what's the value of the loss now?

##
If we're serious about this, we will need to quantitatively understand what the effect on the loss is of varying each and every parameter value independently.  That is, we are going to need to know the _gradient_ of the loss with respect to the parameters:
$$
\nabla L(p)\,.
$$

But as we are reminded below, $p$ is not a simple vector of values.  Its entries are a mix of vectors ($b_1$, $b_2$), matrices ($W_e$, $W_1$, $W_2$), and maybe another day higher order tensors too.  We are going to have to learn how to take gradients of our loss function $L$ with respect to each of those.

In [ ]:
prand

# Gradients done right

In this section we will denote our scalar-valued function by $f$, representing any differentiable mathematical function.  The maths doesn't care if it's a loss function from an AI model, or any other function.  The input variable will be an ordinary column vector $x \in \mathbb{R}^n$ for now, but we'll relax that constraint later.

If you've studied multivariable calculus you will be familiar with the gradient of a function $\nabla f(x)$ defined component-wise: $\nabla f(x)_i = \partial f / \partial x_i$.  This is a perfectly respectable definition of the gradient for functions whose inputs $x$ come from $\mathbb{R}^n$.  But as we've just seen, loss functions in AI come from models with parameters that are not just vectors, but also matrices, or even higher order tensors for more complex models.

In that case it's not at all convenient to work with gradients in component form. While you _could_ just "vectorise" everything (matrices, tensors, the lot) and take gradients the multivariable calculus way, the resulting expressions would be a horrible mess (still correct though).

Instead we prefer an alternative way to define the gradient, that naturally generalises to other vector spaces besides $\mathbb{R}^n$, and "uses the language" of those spaces -- so functions of matrices have gradients expressed in terms of matrices, for example.

Again, if you've studied multivariable calculus you may be familiar with the Taylor expansion of a multivariable function $f$ about a point $x$

$$
f(x+\delta x) = f(x) + \nabla f(x) \cdot \delta x + \textrm{(higher order terms in }\delta x)
$$

which involves the gradient $\nabla f(x)$.  Leveraging this idea turns out to be a suitable way to generalise the definition of gradient to other vector spaces.

Rewriting this as

$$
f(x+\delta x) - f(x) = \nabla f(x) \cdot \delta x + \textrm{(higher order terms in }\delta x)
$$

we recognise the role of the gradient at capturing the "change in $f$ for a small change in $x$".  Specifically, as $\delta x \to 0$ the higher order terms in $\delta x$ matter less and less, and we may write

$$
f(x+\delta x) - f(x) = \nabla f(x) \cdot \delta x + \mathcal{o}(\|\delta x\|),\qquad \delta x \to 0
$$

where the "little-o" notation $\mathcal{o}(\|\delta x\|)$ means "goes to zero faster than $\|\delta x\|$".

The good thing about this interpretation of the gradient is that it naturally generalises to other spaces besides $\mathbb{R}^n$.  We just need a suitable _inner product_ to replace the dot product $\nabla f(x) \cdot \delta x.$

## Inner product spaces
It's worth quickly reviewing the inner products defined on the standard spaces we need for AI. We are of course familiar with the standard inner product on $\mathbb{R}^n$:

The _(Euclidean, or standard) inner product_ on the vector space $\mathbb{R}^n$ is defined by

$$
\langle u, v \rangle = u \cdot v =  u^\top v = \sum_{i=1}^n u_i \,v_i
$$

for all $u, v \in \mathbb{R}^n$.

$\square$

Now for matrices, there is a standard inner product for these too, which is just what you would get if you "vectorised" the matrices and took the Euclidean inner product.  But we much prefer the (equivalent) definition below which uses the language of matrices directly.

Let $\mathbb{R}^{m \times n}$ denote the vector space of real $m \times n$ matrices. The _(Frobenius) inner product_ on this space is defined by

$$
\langle A, B \rangle = \textrm{tr}(A^\top B) = \sum_{i=1}^m \sum_{j=1}^n A_{ij} B_{ij}
$$

for all $A, B \in \mathbb{R}^{m \times n}$.

$\square$

## Properties of trace

Owing to the prominent appearance of the trace in the Frobenius inner product definition, we will often require several properties of the trace which are worth reviewing now.

### Trace of scalar
The first property is trivial and just a statement that under the trace we may identify $1 \times 1$ matrices with scalars, as is our usual practice.

Let $[x]$ be a $1 \times 1$ matrix.  Then $$\textrm{tr}([x]) = x\,.$$

$\square$

### Cyclic permutations
The second result shows that we may cyclically permute a product of matrices without altering the trace.

Let $A_1, A_2, \ldots, A_k$ be matrices (not necessarily square) such that the product

$$
A_1 A_2 \cdots A_k
$$

is square.  Then any cyclic permutation of the factors is also square, and

$$
\textrm{tr}(A_1 A_2 \cdots A_k) = \textrm{tr}(A_2 A_3 \cdots A_1) = \cdots = \textrm{tr}(A_k A_1 \cdots A_{k-1})\,.
$$

$\square$

n.b. Only _cyclic_ permutations are allowed, not just any arbitrary reordering of the factors.

## Gradient defined
We are now in a position to define the gradient of a scalar function with respect to an input from a general inner product space.  As a reminder, our definition should generalise the familiar result from $\mathbb{R}^n$:

$$
f(x+\delta x) - f(x) = \nabla f(x) \cdot \delta x + \mathcal{o}(\|\delta x\|)\,.
$$


Let

$$f: X \to \mathbb{R}$$

be a scalar-valued function defined on a finite-dimensional inner product space $X$.  We say that $f$ is _(Fréchet) differentiable_ at $x \in X$ if there exists an element

$$\nabla_x f \in X\,,$$

called the _gradient_ of $f$ at $x$, such that

$$f(x+\delta x) - f(x) = \langle \nabla_x f, \delta x \rangle + \mathcal{o}(\|\delta x\|)\,.$$

$\square$

While the definition above is the one you'll find in textbooks, in practice it's convenient to use the notation $\textrm{d} x$ in place of $\delta x$ to denote an _infinitesimal_ perturbation, effectively as a shorthand to automatically discard all higher-order terms in $\delta x$.  In that case, we have simply

$$f(x+\textrm{d} x) - f(x) = \langle \nabla_x f, \textrm{d} x \rangle \qquad\textrm{(infinitesimal perturbation)}$$

or even more compactly,

$$\textrm{d} f = \langle \nabla_x f, \textrm{d} x \rangle\,.\qquad\qquad(*)$$

$\square$

So in summary, to find the gradient $\nabla_x f$ we play a little pattern-matching game.  We expand out

$$\textrm{d} f := f(x+\textrm{d} x) - f(x)\,,$$

discard any terms involving higher powers of $\textrm{d} x$, collect up all the remaining terms, and see if we can write it in the form

$$\langle \textrm{[something]}, \textrm{d} x \rangle\,.$$

Whatever the $\textrm{[something]}$ is that makes this work is, by definition, $\nabla _x f$.

## Example 1
Let

$$f = x^\top A x$$

where $x \in \mathbb{R}^n$ and $A \in \mathbb{R}^{n \times n}$.  Find $\nabla_x f$.

Solution:

$$
\require{cancel}
$$

$$
\begin{align*}
\textrm{d} f &= f(x+\textrm{d} x) - f(x)\\
&= (x+\textrm{d} x)^\top A (x+\textrm{d} x) - x^\top Ax\\
&= \cancel{x^\top Ax} + \textrm{d} x^\top Ax + x^\top A\, \textrm{d} x + \xcancel{\textrm{d} x^\top A\, \textrm{d} x} - \cancel{x^\top Ax}\\
&= \textrm{d} x^\top Ax + x^\top A\, \textrm{d} x\\
&= (Ax)^\top\textrm{d} x + (A^\top x)^\top \textrm{d} x\\
&= (Ax + A^\top x)^\top \textrm{d} x\\
&= \langle Ax + A^\top x, \textrm{d} x \rangle
\end{align*}
$$

which is in the form $(*)$

$$
\textrm{d} f = \langle \nabla_x f, \textrm{d} x\rangle
$$

so we identify

$$
\nabla_x f = Ax + A^\top x\,.
$$

## Example 2
Now we consider the same function $f$, but find the gradient with respect to $A$ instead.

Let

$$f = x^\top A x$$

where $x \in \mathbb{R}^n$ and $A \in \mathbb{R}^{n \times n}$.  Find $\nabla_A f$.

Solution:

$$
\begin{align*}
\textrm{d} f &= f(A+\textrm{d} A) - f(A) \\
&= x^\top(A+\textrm{d} A)x - x^\top Ax\\
&= \cancel{x^\top Ax} + x^\top\textrm{d} A\,x - \cancel{x^\top Ax}\\
&= x^\top\textrm{d} A\, x\\
&= \textrm{tr}(x^\top\textrm{d} A\,x)\qquad\textrm{(trace of scalar)}\\
&= \textrm{tr}(xx^\top\textrm{d} A)\qquad\textrm{(cyclic property of trace)}\\
&= \textrm{tr}((xx^\top)^\top\textrm{d} A)\ \ (xx^\top\textrm{ is symmetric)}\\
&= \langle xx^\top, \textrm{d} A \rangle\qquad\ \textrm{(definition of matrix inner product)}\\
\end{align*}
$$

which is in the form $(*)$

$$
\textrm{d} f = \langle \nabla_A f, \textrm{d} A \rangle
$$

so we identify

$$
\nabla_A f = xx^\top\,.
$$

As expected, in Example 1 the gradient $\nabla_x f$ is a vector the size of $x$, while in Example 2 the gradient $\nabla_A f$ is a matrix the size of $A$.

## Now in code

Let's calculate these gradients of this function

$$f = x^\top A x$$

for particular values of $x$ and $A$ to see what we get.

### The function and its gradient with respect to $x$

Here's the code for the function $f$ and the gradient $\nabla_x f$ whose expression we just derived.  We can just pick a random $x_0$ and $A_0$ to try it out.

What to call our gradient function in the code?  We _could_ call it `∇ₓf` to match the mathematics.  But it's more convenient to just call it `∇x`, emphasising what we are taking the gradient _with respect to_ (i.e. $x$), and leaving the function $f$ as being understood from the context.

As expected the result of $\nabla_x f(x_0, A_0)$ is a vector the size of $x_0$.

In [ ]:
f(x,A) = x'*A*x
∇x(x,A) = A*x + A'*x

n = 5
x0 = rand(n)
A0 = rand(n,n)
truegradx = ∇x(x0, A0)

### and gradient with respect to A
And here's the code for the gradient $\nabla_A f(x_0, A_0)$, whose result is a matrix the size of $A_0$.

In [ ]:
∇A(x,A) = x*x'

truegradA = ∇A(x0, A0)

### Numerical verification

We can numerically verify that these values for the gradients are correct using finite differences.  For the vector gradient, each entry should be the value

$$
(\nabla_x f)_i = \frac{\partial f}{\partial x_i} \approx \frac{f(x_0+\varepsilon e_i,A_0) - f(x_0,A_0)}{\varepsilon}
$$

for a suitably small value of $\varepsilon$, where $e_i$ is the vector with a 1 in position $i$ and zeros elsewhere.

Similarly, for the matrix gradient, each entry should be
$$
(\nabla_A f)_{ij} = \frac{\partial f}{\partial A_{ij}} \approx \frac{f(x_0,A_0+\varepsilon E_{ij}) - f(x_0,A_0)}{\varepsilon}
$$

where $E_{ij}$ is the matrix with a 1 in position $(i,j)$ and zeros elsewhere.


##### Unit shifts

Let's confirm our gradients numerically by comparing with these finite difference formulas.  First we need the "unit shift" vector $e_i$ and matrix $E_{ij}$ and a suitably small value of $ε$.

In [ ]:
e(i0) = [i == i0 for i = 1:n]  # 1 in position i0
E(i0, j0) = e(i0) * e(j0)'     # 1 in position (i0,j0)
ε = sqrt(eps())  # square root of machine epsilon

#####Now try out our finite difference vector gradient


In [ ]:
fdgradx = [(f(x0+ε*e(i), A0) - f(x0,A0)) / ε for i = 1:n]

#####and finite difference matrix gradient

In [ ]:
fdgradA = [(f(x0, A0+ε*E(i,j)) - f(x0,A0)) / ε for i = 1:n, j = 1:n]

##### Numerical comparison

If we've done everything right, these finite difference calculations should agree with the true gradients to about $O(\varepsilon) \approx 10^{-8}$.  Here's the comparison, both of which look good.

In [ ]:
maximum(abs, truegradx - fdgradx), maximum(abs, truegradA - fdgradA)

# Back propagation

As we have seen, our model prediction $ \hat{y} = M_p(u)$ is parameterised by a number of different weight matrices and bias vectors.  We have
$$
p = [W_e, W_1, b_1, W_2, b_2]\,.
$$

The value of the loss function $L(y, \hat{y})$ therefore also depends on these parameters, by virtue of its input $\hat{y}$.

It is the the gradients of $L$ with respect to each of the parameters $W_e, W_1, b_1, W_2, b_2$ that we require for training our model.

In the previous section we saw how to find the gradient of a scalar function with respect to a variable belonging to an arbitrary inner product space, so gradients of $L$ with respect to vectors and matrices are fine with us.

But just a moment: evaluating $L$ involves evaluating the model on training data and comparing the output to the ground truth. In general, the model could be quite complex, with many layers upon layers. You might well wonder, to find the gradient of $L$, deep down in the model's inner workings, might we be required to find derivatives of vectors with respect to vectors? Or derivatives of matrices with respect to matrices?

No.

One of the great insights of deep learning is that it's _always_ derivatives of _scalars_ with respect to other variables that we need.  _Never_ derivatives of vectors or matrices with respect to other variables.

At least, that's the case if we are clever in the way that we calculate our derivatives. And the way to be clever about it, is to calculate derivatives by traversing the model _in reverse_.  Since we always have a scalar function (the loss) as the final output, we can ensure we only ever differentiate scalar functions by propagating backwards from there, one layer at a time.  This is the famous **back propagation** algorithm of deep learning.

## Model review

Refer again to our model code, reproduced below for convenience.

Let's summarise what we're doing here.  The variables we calculate, in order, are $X$, $v$, $z_1$, $h_1$, $z_2$ and finally $\hat{y}$. The loss function $L(y, \hat{y})$ is directly a function of $\hat{y}$, but indirectly a function of all the intermediate variables (since they all go towards determining $\hat{y}$). And, as already noted, $L$ is also indirectly a function of the model parameters $W_e$, $W_1$, $b_1$, $W_2$ and $b_2$ since they too are involved in the calculation of the intermediate variables, and hence ultimately, $\hat{y}$.

Hence it makes perfect mathematical sense to ask what is the gradient of $L$ with respect to _every single one of these variables and parameters_.


In [ ]:
# Predict probabilities of the next character
function forward(u, p)
    X = p.We[:, u]               # 1. embedding
    v = vec(X)                   # 2. flatten
    z1 = p.W1 * v  + p.b1        # 3. first dense layer
    h1 = tanh.(z1)               # 4. activation
    z2 = p.W2 * h1 + p.b2        # 5. second dense layer
    ŷ = softmax(z2)              # 6. softmax for probabilities
    return ŷ
end

## Propagating backwards
And that's precisely what the back propagation algorithm does.  Systematically, one by one, it calculates the gradient of $L$ with respect to every variable and parameter in the model.

For our model, this would look like the following steps:
* Calculate $\nabla_{\hat{y}} L$
* Hence, calculate $\nabla_{z_2} L$
* Hence, calculate $\nabla_{W_2} L,$ $\nabla_{b_2} L$ and $\nabla_{h_1} L$
* Hence, calculate $\nabla_{z_1} L$
* Hence, calculate $\nabla_{W_1} L,$ $\nabla_{b_1} L$ and $\nabla_{v} L$
* Hence, calculate $\nabla_{X} L$
* Hence, calculate $\nabla_{W_e} L$

Notice, it's always gradients of $L$ with respect to a vector or matrix. And those gradients all have the same shape as the variable itself.  So in our finished code, for every variable `v`, say, we will have a second variable `∇v` for the gradient of $L$ with respect to $v$.  (There's no need to call the variable `∇ᵥL`; since it's always $L$ we're differentiating it's better to emphasise the variable `v` we're differentiating _with respect to_.)

Let's pick one line of the sequence and see how this works: given  $\nabla_{z_2} L$, calculate $\nabla_{W_2} L,$ $\nabla_{b_2} L$ and $\nabla_{h_1} L$.  The line of code in our model relating these variables is

$$
\texttt{z2 = p.W2 * h1 + p.b2}
$$

or in mathematical notation, dropping subscripts for convenience,

$$
z = W h + b
$$
which is the formula for a standard dense layer with weight matrix $W$ and bias $b$.

We suppose we have available the gradient of $L$ with respect to $z,$ $\nabla_z L$ (which is calculated in the previous step according to the list above) and we will first derive the gradient of $L$ with respect to the weight matrix $W,$ $\nabla_W L$.  So this is a _chain rule_ type of problem:

  * Find $\nabla_W L(z)$ where $z = W h + b$.



Start with what we know, $\nabla_z L$. By definition,

$$
\textrm{d} L = \langle \nabla_z L, \textrm{d} z \rangle\,. \qquad\qquad\qquad(\dagger)
$$

But if the perturbation in $z$ (i.e. $\textrm{d} z$) comes from a perturbation in $W$ (i.e. $\textrm{d} W$), then

$$
\begin{align*}
\textrm{d} z &= z(W+\textrm{d} W) - z(W) \\
&= (W+\textrm{d} W)h + b - (Wh + b) \\
&= \cancel{Wh + b} + \textrm{d} W\, h - \cancel{(Wh + b)}\\
&= \textrm{d} W\, h\,.
\end{align*}
$$

Substitute in $(\dagger)$:

$$
\textrm{d} L = \langle \nabla_z L, \textrm{d} W\, h \rangle\,.
$$

So we've succeeded in expressing the perturbation in $L$ in terms of the perturbation in $W$.  We just need to manipulate this expression to get it in the form

$$
\textrm{d} L = \langle \textrm{[something]}, \textrm{d} W \rangle\,.
$$

Line by line, then:
$$
\begin{align*}
\textrm{d} L &= \langle \nabla_z L, \textrm{d} W\, h \rangle \\
&= (\nabla_z L)^\top \textrm{d} W\, h \qquad\qquad\textrm{(definition of vector inner product)}\\
&= \textrm{tr}\left( (\nabla_z L)^\top \textrm{d} W\, h\right) \qquad\textrm{(trace of scalar)}\\
&= \textrm{tr}\left( h (\nabla_z L)^\top \textrm{d} W\right) \qquad\textrm{(cyclic property of trace)}\\
&= \textrm{tr}\left( (\nabla_z L\, h^\top)^\top \textrm{d} W\right) \quad\textrm{(transpose of product)}\\
&= \langle \nabla_z L\,h^\top, \textrm{d} W \rangle \qquad\quad\textrm{(definition of matrix inner product)}
\end{align*}
$$

And we've done it!  The final line reads

$$
\textrm{d} L = \langle \nabla_z L\,h^\top, \textrm{d} W \rangle
$$

so our pattern-matching has succeeded, and we read straight off:

$$
\nabla_W L = \nabla_z L \, h^\top\,.
$$


This is really just the chain rule for matrix calculus that we've derived.  You can clearly see the parallel to the usual chain rule from calculus.  If you had $z = wh + b$ (all scalars) then the ordinary chain rule would of course read
$$
\frac{\partial L}{\partial w} = \frac{\partial L}{\partial z} \frac{\partial z}{\partial w} = \frac{\partial L}{\partial z} h\,.
$$

The rule we derived above is just this, but adapted to work correctly when the variables are matrices and vectors.  So it's

$$
\nabla_W L = \nabla_z L \, h^\top\,.
$$

We could repeat the same process (exercises!) to derive the formula for the bias gradient

$$
\nabla_b L = \nabla_z L
$$

which is the analogue of the ordinary chain rule

$$
\frac{\partial L}{\partial b} = \frac{\partial L}{\partial z} \frac{\partial z}{\partial b} = \frac{\partial L}{\partial z} \cdot 1 = \frac{\partial L}{\partial z}\,.
$$

These two gradients $\nabla_W L$ and $\nabla_b L$ give us the information we need to understand how adjusting $W$ and $b$, the weights and biases, affect the loss.  If these were the only parameters in the model we'd be done. We could use this gradient information to adjust the weights and biases to drive the loss down.

But of course we're not done, because there are other parameters in the model.  The line we are working on

```
z2 = p.W2 * h1 + p.b2
```

is just one of two dense layers in the model, and there is the embedding layer as well, which all have their own parameters we need gradients for ($W_1$, $b_1$ and $W_e$ respectively).

So we still need to proceed further along the chain of calculations we outlined above to derive these too.  Because there are further gradients to be calculated (corresponding to _earlier_ steps in the model -- we're moving backwards now remember) we're actually not finished with this current step.  We've calculated gradients of the loss with respect to the _parameters_ $W_2$ and $b_2$.  We also need to calculate the gradient with respect to the _data_ $h_1$.

Back to the mathematical form

$$
z = Wh + b
$$

still need to derive $\nabla_h L$.  It's this gradient that will propagate backwards from here, enabling further calculations of gradients from the layer before.  So once more then, now perturbing $h$:

$$
\begin{align*}
\textrm{d} z &= z(h+\textrm{d} h) - z(h) \\
&= W(h + \textrm{d} h) + b - (Wh + b) \\
&= \cancel{Wh + b} + W \textrm{d} h - \cancel{(Wh + b)}\\
&= W \textrm{d} h
\end{align*}
$$

Substitute into
$$
\textrm{d} L = \langle \nabla_z L, \textrm{d} z \rangle
$$
and rearrange:

$$
\begin{align*}
\textrm{d} L &= \langle \nabla_z L, W \textrm{d} h \rangle \\
&= (\nabla_z L)^\top W \textrm{d} h \qquad\quad\textrm{(definition of vector inner product)}\\
&= (W^\top\, \nabla_z L)^\top \textrm{d} h \qquad\textrm{(transpose of product)}\\
&= \langle W^\top\, \nabla_z L, \textrm{d} h \rangle \qquad\textrm{(definition of vector inner product)}
\end{align*}
$$

so we can read off:

$$
\nabla_h L = W^\top\, \nabla_z L\,.
$$

## The famous formulas

That's the full set of gradients for this dense layer now.  We have derived

$$
\nabla_W L = \nabla_z L \, h^\top, \quad \nabla_b L = \nabla_z L \quad \textrm{and} \quad \nabla_h L = W^\top\, \nabla_z L\,.
$$

These are very famous formulas, since they paved the way for the deep learning revolution by enabling efficient training using back propagation.  And now you understand them completely.

# Conclusion

In this lesson we learned:

* the idea of a loss function to quantify the mismatch between the model's predictions and the ground truth
* the specific cross entropy loss function
* how gradients describe the sensitivity of the loss to each parameter
* matrix calculus, i.e. gradients defined on inner product spaces
* how finite difference checks can be used to verify gradient implementations
* the gradient formulas for dense layers, including weights, biases, and inputs
* the famous formulas for back propagation

In the next lesson we will use this knowledge to develop the full _backward pass_ for the model, and also explore aspects of automatic differentiation.
